In [1]:
from langgraph.graph import StateGraph
from typing import TypedDict
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from chat_memory import get_chat_history
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage
from video_processing import analyze_video
from vectorstore_setup import clean_classification_text
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END
import tempfile

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="path/to/your/chroma_db", embedding_function=embeddings)

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")



# when a query comes in, use this model to embed the query text so you can compare it against the vectors already stored.
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/chroma_db", 
                                    embedding_function=embeddings)


# Whisper speach to text 
from openai import OpenAI
client = OpenAI()




In [2]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Your job is to classify user queries into two buckets: 
     - memory
     - memory and vectorstore
     
     *Guidelines*
     Questions regarding exercises, exercise form, or technique: vectorstore & memory
     General question or statements: memory
   
     *Respond to queries with only the correct class*
     Examples: 
     User: how should I grip the bar for bench press?
     Agent: vectorstore & memory

     User: what muscles should I feel during the Romanian dead lift?
     Agent: vectorstore & memory

     User: How's this? 
     Agent: vectorstore & memory

     User: Is this good squat technique? 
     Agent: vectorstore & memory

     User: Was my bar path better?
     Agent: vectorstore & memory

     User: thakns for your help!
     Agent: memory

"""),

     ("human", "{query}")
     
])

llm = ChatOpenAI(model='gpt-4o')

output_parser = StrOutputParser()

router_chain = router_prompt | llm | output_parser


In [ ]:
# Define the state

# the state is a dictionary

class GraphState(TypedDict):

    session_id: int 

    user_query: str

    user_video: str

    classification_image: str

    classified_keywords: str

    top_k_chunks: str

    encoded_images: list[str]

    response: str


workflow = StateGraph(GraphState) 

In [4]:
workflow = StateGraph(GraphState) 

In [5]:
def video_encoder_node(state: GraphState):
    user_video = state["user_video"]
    user_video_encoded = analyze_video(filepath_in=user_video, frame_count=15, max_seconds=10) # encodes video
    encoded_images = [] # creates a list of encoded images for LLM
    for images in user_video_encoded:
        encoded_images.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{images}"}})
        classification_image = user_video_encoded[0] 
    
    return {"classification_image": classification_image, "encoded_images": encoded_images}


In [6]:
# video classification router NODE

def video_classification_node(state: GraphState):
    classification_image = state["classification_image"]

    router_llm = ChatOpenAI(model='gpt-4o')

    response = router_llm.invoke([

    {"role": "user", "content": 

    [{"type": "text", "text":  """Your job is to analyze images of users working out for proper form, and list the key checkpoints of their to body evaluate. 
    Give me ONLY the bodypart checkpoints. Do NOT include evaluation suggestions. Do NOT include an intro sentence. 
    Output format should be exactly the example below.
    **Example**
    Overhead press

    1. Feet & base
    2. Glutes & legs
    3. Core & Ribcage
    4. Shoulder position
    5. Bar path
    6. Head & Neck
    7. Lockout position
    8. Tempo and control
    """},


        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{classification_image}"}}
        ]}


    ])
    print(response.text)

    classified_keywords = clean_classification_text(response)

    return {"classified_keywords": classified_keywords}


In [7]:
def vector_db_node(state: GraphState):
    user_query = state["user_query"]
    classified_keywords = state.get("classified_keywords", "")
    retrieval_query = classified_keywords

    if classified_keywords:
        results_user = vectorstore.similarity_search(user_query, k=5)
        results_video = vectorstore.similarity_search(retrieval_query, k=5)
        results = results_user + results_video

    if not classified_keywords: 
        results_user = vectorstore.similarity_search(user_query, k=5)
        results = results_user

    unique_results = []
    seen = set()

    for r in results:
        content = r.page_content
        if content not in seen:
            seen.add(content)
            unique_results.append(r)


    top_k_chunks = "\n".join([r.page_content for r in unique_results])

    return {"top_k_chunks": top_k_chunks}

In [20]:
def response_generator(state: GraphState):
    top_k_chunks = state['top_k_chunks']
    encoded_images = state.get("encoded_images", [])
    session_id = state["session_id"]
    user_query = state["user_query"]

    prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 
    Inspect each image CLOSELY and arefuly for problems or issues related to best practices in exercise form. Help the user diagnose their incorrect form. 
    Be specific about what you observe.

    # ANSWER CONTEXT
    Use ONLY the following context when answering a user: 
        
    ---   
    {top_k_chunks}
    ---
    if the query or image isn't in context, reply, 'I don't have expert coaching content for this exercise yet. 
    Currently I can analyze: bench press, overhead press, incline bench press...'"
    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])



    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query},
        *encoded_images,   # <- your list of {"type":"image_url",...}
    ])

    llm = ChatOpenAI(model='gpt-5',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg], "top_k_chunks": top_k_chunks},
        config={"configurable": {"session_id": session_id}}
    )


    return {"response": response}
    

In [9]:
def chat_memory(state: GraphState):
    user_query = state["user_query"]
    session_id = state["session_id"]
    
    prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 


    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])

    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query}
    ])

    llm = ChatOpenAI(model='gpt-4o',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg]},
        config={"configurable": {"session_id": session_id}}
    )

    return {"response": response}
    


# Router function

In [10]:
def route_query(state: GraphState):
    # first check: is there a video?
    if state["user_video"]:
        return "video_encoder"
    
    # no video — ask the LLM which path
    route = router_chain.invoke({"query": state["user_query"]})
    route = route.lower().strip()
    
    if "vectorstore" in route:
        return "vector_db"
    else:
        return "chat_memory"

# Audio transcription function

In [ ]:
def transcribe_audio(audio_path):
    audio_file = open(audio_path, "rb") # rb = read binary. audio files are binary data (raw bytes). this tells python gto open as binary instaed of text
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

    return transcription.text
    

user_query = transcribe_audio("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_voice_notes/bench_press_voice_note.m4a")
print(user_query)

In [12]:
# conditional edge
workflow.add_conditional_edges(START, route_query)

# add nodes
workflow.add_node("video_encoder", video_encoder_node)
workflow.add_node("video_classification_router", video_classification_node)
workflow.add_node("vector_db", vector_db_node)
workflow.add_node("response_generator", response_generator)
workflow.add_node("chat_memory", chat_memory)



# connect them
workflow.add_edge("chat_memory", END)
workflow.add_edge("video_encoder", "video_classification_router")
workflow.add_edge("video_classification_router", "vector_db")
workflow.add_edge("vector_db", "response_generator")
workflow.add_edge("response_generator", END)

app = workflow.compile()


In [13]:
result = app.invoke({
    "session_id": 123,
    "user_query": "how's my form?",
    "user_video": "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_workout_videos/bench_press/bench_07.mov"
})

print(result["response"])

Frames processed: 15, (10s cap)
Saved to processed-images/bench_07
Video name: bench_07
**Bench Press**

1. Feet & base
2. Glutes & legs
3. Lower back & arch
4. Core engagement
5. Shoulder position
6. Grip & bar path
7. Head & neck
8. Elbow alignment
9. Tempo and control
Thanks for the video—this is a flat barbell bench press. Here’s what I’m seeing and how to fix it.

What looks good
- You’ve set a solid arch and you’re touching the bar to the chest each rep.

Key fixes
- Feet/leg drive: Your feet are too far forward and too wide from the bench, so your shins aren’t near vertical and you can’t drive well. Lower one foot at a time and tuck them back as far as you comfortably can, heels planted, legs close to the bench when viewed from the front. Use the heels as the base of your drive.
- Hips: I see your hips start to rise as you press. Keep your butt touching the bench at all times. Use leg drive to reinforce the arch, not to bridge your hips up.
- Upper back tightness: Pinch your sho

In [14]:

# result = app.invoke({
#     "session_id": 123,
#     "user_query": user_query,
#     "user_video": ""
# })

# print(result["response"])

In [15]:
# result = app.invoke({
#     "session_id": 123,
#     "user_query": "thanks for your help today!",
#     "user_video": ""
# })

# print(result["response"])

In [16]:
# result = app.invoke({
#     "session_id": 123,
#     "user_query": "can you tell me again how to make it easier?",
#     "user_video": ""
# })

# print(result["response"])

In [ ]:
# def correctness_evaluator(run, example):
#     generated = run.outputs["answer"]
#     reference = example.outputs["answer"]
#     question = example.inputs["user_query"]

#     prompt = f"""You are evaluating a fitness coaching AI assistant.
# Your job is to assess the quality of its response against a reference answer.

# Question: {question}
# Reference answer: {reference}
# Generated answer: {generated}

# Scoring criteria:

# Score 0 — The generated answer contains incorrect or potentially harmful advice that contradicts the reference answer. This is the worst outcome.
# Score 1 — The generated answer is incomplete or omits important details from the reference, but contains nothing incorrect or harmful.
# Score 2 — The generated answer is correct and captures the key facts from the reference answer.

# Important: Penalize incorrect advice more harshly than omission. A response that says nothing is better than one that says something wrong.

# Reply with only the number: 0, 1, or 2."""

#     from openai import OpenAI
#     openai_client = OpenAI()
#     result = openai_client.chat.completions.create(
#         model="gpt-4o",
#         messages=[{"role": "user", "content": prompt}]
#     )
#     try: 
#         score_raw = result.choices[0].message.content.strip()
#         score = int(''.join(filter(str.isdigit, score_raw))[0])
#     except:
#         score = -1
#     return {"key": "correctness", "score": score / 2} # normalize the score so it's <= 1

In [ ]:
# def hallucination_evaluator(run, example):
#     generated = run.outputs["answer"]
#     contexts = run.outputs["contexts"]
    
#     prompt = f"""You are evaluating a fitness coaching AI assistant for hallucinations.
# You are an expert data labeler evaluating model outputs for hallucinations. Your task is to assign a score based on the following rubric:

# <Rubric>
#   A response without hallucinations:
#   - Contains only verifiable facts that are directly supported by the input context
#   - Makes no unsupported claims or assumptions
#   - Does not add speculative or imagined details
#   - Maintains perfect accuracy in dates, numbers, and specific details
#   - Appropriately indicates uncertainty when information is incomplete
# </Rubric>

# <Instructions>
#   - Read the input context thoroughly
#   - Identify all claims made in the output
#   - Cross-reference each claim with the input context
#   - Note any unsupported or contradictory information
#   - Consider the severity and quantity of hallucinations
# </Instructions>

# <Reminder>
#   Focus solely on factual accuracy and support from the input context. Do not consider style, grammar, or presentation in scoring. A shorter, factual response should score higher than a longer response with unsupported claims.
# </Reminder>

# Use the following context to help you evaluate for hallucinations in the output:

# Retrieved context:
# {contexts}

# Generated answer:
# {generated}

# Scoring criteria:

# Score 0 — The answer contains claims that are not supported by or directly contradict the retrieved context. This includes inventing observations, giving advice not found in the context, or stating things as fact without context support.
# Score 1 — The answer is mostly supported by the context but includes minor unsupported details.
# Score 2 — Every claim in the answer is directly supported by the retrieved context.

# Important: A shorter, grounded answer should score higher than a longer answer with unsupported claims.

# Reply with only the number: 0, 1, or 2."""

#     from openai import OpenAI
#     openai_client = OpenAI()
#     result = openai_client.chat.completions.create(
#         model="gpt-4o",
#         messages=[{"role": "user", "content": prompt}]
#     )
#     try:
#         score_raw = result.choices[0].message.content.strip()
#         score = int(''.join(filter(str.isdigit, score_raw))[0])
#     except:
#         score = None
#     return {"key": "hallucination", "score": score}

In [ ]:
from uuid import uuid4

def run_rag_pipeline(inputs: dict, attachments: dict) -> dict:
    # grab video from attachments in LangSmith
    video_name = list(attachments.keys())[0] 
    # read the video as raw bytes
    video_bytes = attachments[video_name]["reader"].read()
    
    # save to temp file because analyze_video expects a filepath
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
        # . delete=False means "don't delete this file when we're done writing" because we still need it for the pipeline.
        tmp.write(video_bytes)
        tmp_path = tmp.name
    
    state = {
        "user_query": inputs["user_query"],
        "session_id": f"eval-{uuid4()}",
        "classified_keywords": "",
        "encoded_images": [],
        "user_video": tmp_path
    }
    
    state = {**state, **video_encoder_node(state)}
    state = {**state, **video_classification_node(state)}
    state = {**state, **vector_db_node(state)}
    state = {**state, **response_generator(state)}

    print("STATE KEYS:", state.keys())
    print("RESPONSE:", state.get("response", "NOT FOUND"))
    
    return {"answer": state["response"],
            "contexts": state["top_k_chunks"]}

In [ ]:
# from langsmith import Client
# from langsmith.evaluation import evaluate

# langsmith_client = Client()

# os.environ["LANGCHAIN_TRACING_V2"] = "false" 
# # stops LangSmith from trying to log all the big intermediate data
# # evaluate() function will still capture your inputs, outputs, and evaluator scores
# # it just won't try to upload the full trace with all those heavy frames.

# results = evaluate(
#     run_rag_pipeline,
#     data="Image assessment response accuracy v1",
#     evaluators=[correctness_evaluator, hallucination_evaluator],
#     experiment_prefix="vision-faithfulness-rag-v6-hallucination"
# )

View the evaluation results for experiment: 'vision-faithfulness-rag-v6-hallucination-6f876117' at:
https://smith.langchain.com/o/5dffb1fc-82e5-4000-a9a5-d9c923e0f73c/datasets/1be4093e-6f7d-4021-a94b-d6cf76c22b6a/compare?selectedSessions=01e35544-02b1-4bd9-bbf4-111fadbc2d13




0it [00:00, ?it/s]

Frames processed: 15, (10s cap)
Saved to processed-images/tmpje1ffbrd
Video name: tmpje1ffbrd
**Bench Press**

1. Feet & base
2. Glutes & legs
3. Lower back & arch
4. Shoulder position
5. Elbow alignment
6. Bar path
7. Grip
8. Head & neck
9. Tempo and control
STATE KEYS: dict_keys(['user_query', 'session_id', 'classified_keywords', 'encoded_images', 'user_video', 'classification_image', 'top_k_chunks', 'response'])
RESPONSE: Nice work getting consistent reps. Here’s what I’m seeing on your bench and what to tweak for tighter, stronger reps:

What looks good
- Bar path is fairly straight and you’re controlling the descent.
- Head, upper back and glutes are staying on the bench.

Key fixes
- Foot position/leg drive: Your feet are a bit forward and wide. Slide each foot back as far as you comfortably can while keeping them flat on the floor. Keep the legs closer to the bench when viewed from the front. Stay planted the whole set—no floppy feet. This will lock in your base and make pressin

Error running target function: [Errno 2] No such file or directory: '/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/processed/processed-images/tmp8eb1i00q_14.jpg'
Traceback (most recent call last):
  File "/opt/miniconda3/envs/fitness-form-coach/lib/python3.10/site-packages/langsmith/evaluation/_runner.py", line 1903, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/opt/miniconda3/envs/fitness-form-coach/lib/python3.10/site-packages/langsmith/run_helpers.py", line 758, in wrapper
    function_result = run_container["context"].run(
  File "/var/folders/2n/c95ntf5d1wx1_vrd1b3ljbkw0000gn/T/ipykernel_62315/1058491311.py", line 23, in run_rag_pipeline
    state = {**state, **video_encoder_node(state)}
  File "/var/folders/2n/c95ntf5d1wx1_vrd1b3ljbkw0000gn/T/ipykernel_62315/436431845.py", line 3, in video_encoder_node
    user_video_encoded = analyze_video(filepath_in=user_video, frame_count=15, max_seconds=10) # encodes video
  File "/Users/chandlersho

Frames processed: 15, (10s cap)
Saved to processed-images/tmp8eb1i00q
Video name: tmp8eb1i00q
Frames processed: 15, (10s cap)
Saved to processed-images/tmpbf_afabe
Video name: tmpbf_afabe
**Bench Press**

1. Feet & base
2. Glutes & lower back
3. Core & Ribcage
4. Elbow & Arm position
5. Wrist alignment
6. Bar path
7. Shoulder position
8. Head & Neck
STATE KEYS: dict_keys(['user_query', 'session_id', 'classified_keywords', 'encoded_images', 'user_video', 'classification_image', 'top_k_chunks', 'response'])
RESPONSE: Thanks for the clips—this is your bench press. Here’s what I’m seeing and how to tighten it up.

What looks good
- You’re touching the bar to your chest each rep and you’ve got a visible arch.
- Collars on and the load looks even. Nice.

Biggest fixes
1) Elbow path
- Your elbows look pretty flared at the bottom. That’s a common error and tends to stress the shoulder and force a straight up‑and‑down bar path.
- On the descent, drop your elbows to roughly a 45° angle relative 